# ParsBERT: Sentence Embeddings + t-SNE by Topic (Negative Samples)

This notebook mirrors `parsbert_embeddings.ipynb`, but for the **negative** (non-conditional)
examples in `persian_negatives_bio.jsonl`:
1. Loads **ParsBERT** (`HooshvareLab/bert-fa-zwnj-base`).
2. Computes mean-pooled sentence embeddings for every negative sample.
3. Reduces them to 2D with t-SNE.
4. Colors the plot by topic, and labels one representative example sentence per topic cluster.

**Note on "topic":** unlike `persian_conditionals_bio.jsonl`, the negative-example generation
notebook (`generate_negatives.ipynb`) *does* save the sampled topic directly on each record (the
`topic` field), so there is no need for the zero-shot nearest-topic trick used in the conditionals
notebook — we use the ground-truth topic tags as-is.

**Requirements:** `pip install transformers torch scikit-learn matplotlib`
(optionally `pip install arabic-reshaper python-bidi` for correctly shaped Persian text *inside*
the plot — see the "Plot" section).


In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.manifold import TSNE
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

from persian_display import display_persian

## Load ParsBERT

In [ ]:
MODEL_NAME = "HooshvareLab/bert-fa-zwnj-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print(f"Loaded {MODEL_NAME} on {device}, hidden size = {model.config.hidden_size}")

## Sentence embeddings via mean pooling

ParsBERT (like base BERT) has no dedicated sentence-embedding head, so we mean-pool the last
hidden state over real (non-padding) tokens, weighted by the attention mask — the standard,
model-agnostic way to turn token embeddings into a single sentence vector for a plain encoder.

In [ ]:
@torch.no_grad()
def embed_texts(texts, batch_size=32, max_length=64):
    """Mean-pooled ParsBERT sentence embeddings for a list of texts, shape (len(texts), hidden)."""
    all_embeddings = []
    for start in tqdm(range(0, len(texts), batch_size)):
        batch = texts[start : start + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt"
        ).to(device)
        output = model(**encoded)
        token_embeddings = output.last_hidden_state  # (batch, seq, hidden)
        mask = encoded["attention_mask"].unsqueeze(-1).float()  # (batch, seq, 1)
        summed = (token_embeddings * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        mean_pooled = summed / counts
        all_embeddings.append(mean_pooled.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)

## Load the dataset

In [ ]:
DATA_PATH = Path("persian_negatives_bio.jsonl")

examples = [json.loads(line) for line in DATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
texts = [ex["text"] for ex in examples]

print(f"Loaded {len(examples)} examples from {DATA_PATH}")

## Embed all sentences

Set `SAMPLE_SIZE` to a smaller number for a quick run; `None` embeds the whole dataset (a few
minutes on CPU for ~1000 short sentences with `bert-fa-zwnj-base`).

In [ ]:
SAMPLE_SIZE = None  # e.g. 300 for a faster run

if SAMPLE_SIZE is not None:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(examples), size=min(SAMPLE_SIZE, len(examples)), replace=False)
    sample_examples = [examples[i] for i in idx]
else:
    sample_examples = examples

sample_texts = [ex["text"] for ex in sample_examples]
sentence_embeddings = embed_texts(sample_texts)
print("sentence_embeddings shape:", sentence_embeddings.shape)

## Topic per sentence

Unlike the conditionals notebook, `generate_negatives.ipynb` saves the sampled `topic` field
directly on each record, so we read it off instead of inferring it from embedding similarity.
Topics are the same 15 phrases sampled in both generation notebooks; we keep the same fixed
ordering (and English glosses) for consistent coloring/legend order across both notebooks.

In [ ]:
# (Persian phrase as stored in the "topic" field, English gloss used for plot labels)
TOPICS = [
    ("اقتصاد و قیمت‌ها", "Economy & prices"),
    ("آب‌وهوا", "Weather"),
    ("سفر", "Travel"),
    ("تحصیل و دانشگاه", "Education & university"),
    ("خانواده و روابط", "Family & relationships"),
    ("تکنولوژی", "Technology"),
    ("سلامت", "Health"),
    ("ورزش", "Sports"),
    ("محیط زیست", "Environment"),
    ("کار و شغل", "Work & jobs"),
    ("غذا و آشپزی", "Food & cooking"),
    ("ترافیک شهری", "Urban traffic"),
    ("دوستی", "Friendship"),
    ("سیاست", "Politics"),
    ("موسیقی و هنر", "Music & art"),
]
topic_fa_to_en = dict(TOPICS)
topic_en = [en for _fa, en in TOPICS]

sample_topics_fa = [ex.get("topic", "<none>") for ex in sample_examples]
inferred_topics_en = [topic_fa_to_en.get(fa, fa) for fa in sample_topics_fa]

print("Topic counts:")
for topic, count in Counter(inferred_topics_en).most_common():
    print(f"  {topic:<25} {count}")

## t-SNE

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=min(30, max(5, len(sample_texts) // 10)),
    random_state=42,
    init="pca",
)
coords_2d = tsne.fit_transform(sentence_embeddings)
print("coords_2d shape:", coords_2d.shape)

## One representative example per topic

For each topic that appears, we pick the sentence whose embedding is closest (by cosine
similarity) to the mean embedding of all sentences with that topic label (its most "typical"
example) and print it, alongside its t-SNE coordinates so it can be cross-checked against the plot
below.

In [ ]:
def cosine_similarity_matrix(a, b):
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a_norm @ b_norm.T


inferred_topics_en_arr = np.array(inferred_topics_en)
representative_idx = {}
representative_sim = {}
for topic in set(inferred_topics_en):
    mask = inferred_topics_en_arr == topic
    topic_mean = sentence_embeddings[mask].mean(axis=0, keepdims=True)
    sims = cosine_similarity_matrix(sentence_embeddings[mask], topic_mean)[:, 0]
    local_idx = np.where(mask)[0]
    best_local = sims.argmax()
    representative_idx[topic] = local_idx[best_local]
    representative_sim[topic] = sims[best_local]

print(f"Representative example per topic ({len(representative_idx)} topics present):\n")
for topic in topic_en:
    if topic not in representative_idx:
        continue
    i = representative_idx[topic]
    x, y = coords_2d[i]
    display_persian(f"[{topic}] (sim={representative_sim[topic]:.2f}, xy=({x:.1f}, {y:.1f}))  {sample_texts[i]}")

## Plot

Note: 15 topics is more categories than a strict CVD-safe categorical palette can guarantee
pairwise-distinguishable in an all-pairs scatter (that safely caps out around 3-4 hues without
secondary encoding). For this exploratory plot we use `tab20`, a qualitative colormap designed for
up to 20 categories, with hues assigned in the fixed topic order above and a legend always shown.
If you need a fully accessible version, facet into small multiples (one panel per topic) instead.

The representative sentence for each topic is annotated directly on its point. matplotlib does not
apply Arabic/Persian glyph shaping or bidi reordering on its own (the same issue as printing raw
Persian text to a non-bidi-aware terminal), so annotations use `arabic_reshaper` + `python-bidi` to
pre-shape the text when those optional packages are installed; otherwise they fall back to the raw
string, which may render with disconnected letters / wrong left-right order.

In [ ]:
try:
    import arabic_reshaper
    from bidi.algorithm import get_display

    def shape_for_plot(text):
        return get_display(arabic_reshaper.reshape(text))
except ImportError:
    def shape_for_plot(text):
        return text


def truncate(text, max_chars=28):
    return text if len(text) <= max_chars else text[:max_chars].rstrip() + "…"


cmap = plt.get_cmap("tab20")
topic_colors = {topic: cmap(i / max(1, len(topic_en) - 1)) for i, topic in enumerate(topic_en)}

fig, ax = plt.subplots(figsize=(12, 9))
for topic in topic_en:
    mask = np.array(inferred_topics_en) == topic
    if not mask.any():
        continue
    ax.scatter(
        coords_2d[mask, 0],
        coords_2d[mask, 1],
        s=24,
        color=topic_colors[topic],
        label=topic,
        alpha=0.8,
        edgecolors="none",
    )

for topic, i in representative_idx.items():
    x, y = coords_2d[i]
    ax.scatter([x], [y], s=90, facecolors="none", edgecolors=topic_colors[topic], linewidths=1.5, zorder=5)
    label = shape_for_plot(truncate(sample_texts[i]))
    ax.annotate(
        label,
        xy=(x, y),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=7,
        color=topic_colors[topic],
        ha="left",
        va="bottom",
    )

ax.set_title("t-SNE of ParsBERT sentence embeddings (negative samples), colored by topic")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9, title="Topic (ground truth)")
fig.tight_layout()
plt.show()